In [0]:
import requests
import json

# Extração controlada via API para análise de esquema sem sobrecarga de rede
url = "https://api.openalex.org/works?per-page=5&mailto=depaulasilvadiego297@gmail.com"
response = requests.get(url)

if response.status_code == 200:
    dados = response.json().get("results", [])
    print(f"Sucesso: {len(dados)} trabalhos recuperados para análise exploratória.")
else:
    raise Exception(f"Erro na API: {response.status_code}")

# Mitigação prévia de colisão de tipos no PySpark descartando atributos ruidosos
for item in dados:
    item.pop("abstract_inverted_index", None)

# Persistência física temporária para adequação ao modelo de computação Serverless
caminho_temp = "/tmp/amostra_openalex.json"
with open(caminho_temp, "w", encoding="utf-8") as f:
    json.dump(dados, f)

# Ingestão em memória utilizando o motor distribuído do Spark
df_amostra = spark.read.json(caminho_temp)

# Inspeção da árvore de metadados para evidenciar a estrutura semiestruturada
print("\n--- ESQUEMA ESTRUTURAL DO OPENALEX ---")
df_amostra.printSchema()

# Validação visual das matrizes que exigirão desaninhamento na Camada Prata
print("\n--- AMOSTRA DOS CAMPOS ANINHADOS ---")
display(df_amostra.select("id", "title", "publication_year", "authorships", "referenced_works"))